# <font color="#418FDE" size="6.5" uppercase>**Sequenzmodelle**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Bereiten Sequenzfenster und Baselines für zeitliche Vorhersage- oder Klassifikationsaufgaben vor. 
- Trainieren kleine Conv1D-, SimpleRNN- und LSTM- oder GRU-Modelle CPU-freundlich. 
- Vergleichen Sequenznetze mit klassischen Signalmodellen anhand zeitlicher Fehleranalysen. 


## **1. Sequenzen vorbereiten**

### **1.1. Batchformen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_01_01.jpg?v=1787665973" width="250">



>* Zeitreihen werden in feste Fenster geteilt
>* Batches zeigen zeitliche Muster und Merkmale

>* Fensterlänge passend zur Prozessdynamik wählen
>* Überlappung und Ziel je Aufgabe festlegen

>* Einheitliche Fenster sichern korrekte Batchverarbeitung
>* Zukunftsdaten vermeiden, Experimente sauber vergleichen



In [ ]:
#@title Python-Code - Batchformen verstehen

# Dieses Beispiel zeigt Batchformen für Sequenzfenster.
# Wir zerlegen ein Signal in überlappende Fenster.
# Am Ende siehst du Eingaben und Zielwerte.

import numpy as np
import matplotlib.pyplot as plt

# Ein kleines Signal steht für Messwerte über die Zeit.
time_steps = np.arange(12)
signal = np.array([18, 19, 21, 20, 22, 24, 23, 25, 27, 26, 28, 30])

# Jedes Fenster nutzt vier vergangene Zeitschritte.
window_length = 4
horizon = 1

# Die Anzahl möglicher Fenster ergibt sich aus Länge und Zielabstand.
window_count = len(signal) - window_length - horizon + 1
if window_count <= 0:
    raise ValueError("Das Signal ist zu kurz für diese Fensterlänge.")

# X sammelt Eingabefenster, y den jeweils nächsten Wert.
windows = []
targets = []

# Überlappende Fenster erzeugen mehrere Trainingsbeispiele.
for start in range(window_count):
    end = start + window_length
    windows.append(signal[start:end])
    targets.append(signal[end + horizon - 1])

# Sequenzmodelle erwarten oft drei Dimensionen.
X = np.array(windows).reshape(window_count, window_length, 1)
y = np.array(targets)

# Ein Batch ist hier eine kleine Auswahl von Beispielen.
batch_size = 3
batch_X = X[:batch_size]
batch_y = y[:batch_size]

print(f"Gesamtes Signal: {len(signal)} Zeitschritte")
print(f"X-Form: {X.shape} = Beispiele, Zeitfenster, Merkmale")
print(f"y-Form: {y.shape} = ein Zielwert pro Fenster")
print(f"Batch-Form: {batch_X.shape}")
print(f"Erstes Fenster: {batch_X[0, :, 0].tolist()} -> Ziel {batch_y[0]}")

# Die Grafik markiert das erste Eingabefenster und sein Ziel.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time_steps, signal, marker="o", label="Messsignal")
ax.axvspan(0, window_length - 1, color="tab:blue", alpha=0.15, label="Eingabe")
ax.scatter(window_length, y[0], color="tab:red", s=80, label="Zielwert")
ax.set_title("Vom Signal zur Batchform")
ax.set_xlabel("Zeitschritt")
ax.set_ylabel("Messwert")
ax.legend()
plt.show()



### **1.2. Naive Vergleichsmodelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_01_02.jpg?v=1787665969" width="250">



>* Letzter Wert als einfache Prognose
>* Komplexe Modelle müssen klar besser sein

>* Mehrheitsklasse, letzte Klasse oder Regeln nutzen
>* Unausgewogene Daten mit Baselines kritisch prüfen

>* Baselines fair und ohne Zukunftsdaten vergleichen
>* Fehler zeitlich analysieren, nicht nur mitteln



In [ ]:
#@title Python-Code - Naive Vergleichsmodelle

# Dieses Beispiel vergleicht einfache Zeitreihen-Baselines.
# Sequenzfenster liefern faire Eingaben für Prognosen.
# Die Fehler zeigen Grenzen naiver Modelle.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen eine kleine, deterministische Zeitreihe.
rng = np.random.default_rng(42)
time_steps = np.arange(80)

trend = 0.04 * time_steps
season = 1.5 * np.sin(time_steps / 6)
noise = rng.normal(0, 0.25, size=time_steps.size)

values = 10 + trend + season + noise
window_size = 6

# Jedes Fenster nutzt nur vergangene Werte.
windows = []
targets = []

for start in range(len(values) - window_size):
    windows.append(values[start:start + window_size])
    targets.append(values[start + window_size])

windows = np.array(windows)
targets = np.array(targets)

# Eine einfache Prüfung schützt vor falschen Fenstergrößen.
if windows.shape[0] != targets.shape[0]:
    raise ValueError("Fenster und Zielwerte passen nicht zusammen.")

# Die naive Prognose übernimmt den letzten Fensterwert.
last_value_predictions = windows[:, -1]
mean_window_predictions = windows.mean(axis=1)

last_mae = np.mean(np.abs(targets - last_value_predictions))
mean_mae = np.mean(np.abs(targets - mean_window_predictions))

print(f"Anzahl Sequenzfenster: {windows.shape[0]}")
print(f"Fensterlänge: {window_size} Zeitpunkte")
print(f"MAE letzte-Wert-Baseline: {last_mae:.3f}")
print(f"MAE Fenster-Mittelwert-Baseline: {mean_mae:.3f}")
print("Kleinerer MAE bedeutet bessere Vorhersage.")

# Wir zeigen die Zielwerte und beide Baselines.
fig, ax = plt.subplots(figsize=(9, 4))
plot_index = np.arange(targets.size)

ax.plot(plot_index, targets, label="Echter nächster Wert", linewidth=2)
ax.plot(plot_index, last_value_predictions, label="Letzter Wert", alpha=0.8)
ax.plot(plot_index, mean_window_predictions, label="Fenster-Mittelwert", alpha=0.8)

ax.set_title("Naive Vergleichsmodelle für Sequenzfenster")
ax.set_xlabel("Fenster-Index")
ax.set_ylabel("Messwert")
ax.legend()

plt.show()



### **1.3. Zeitliche Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_01_03.jpg?v=1787665971" width="250">



>* Zeitreihen chronologisch statt zufällig aufteilen
>* Nur Vergangenes zum Vorhersagen nutzen

>* Datenleckage macht Bewertungen zu optimistisch
>* Fenster, Ziele und Skalierung sauber trennen

>* Split passend zu Aufgabe und Saison wählen
>* Baselines auf denselben Zeiträumen vergleichen



In [ ]:
#@title Python-Code - Zeitliche Splits

# Dieses Beispiel zeigt saubere zeitliche Splits.
# Fenster bleiben vollständig in ihrem Datenabschnitt.
# Die Grafik macht Datenleckage sichtbar.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen eine kleine synthetische Zeitreihe.
time_index = np.arange(60)
values = 20 + 0.15 * time_index + 2 * np.sin(time_index / 4)

# Die Grenzen folgen strikt der zeitlichen Reihenfolge.
train_end = 36
validation_end = 48
window_size = 6

# Diese Funktion baut Fenster ohne Abschnittsgrenzen zu überschreiten.
def make_windows(series, start, end, window_size):
    rows = []
    for target_time in range(start + window_size, end):
        input_start = target_time - window_size
        rows.append((input_start, target_time, series[target_time]))
    return pd.DataFrame(rows, columns=["input_start", "target_time", "target"])

# Jedes Fenster wird nur innerhalb seines Splits erzeugt.
train_windows = make_windows(values, 0, train_end, window_size)
validation_windows = make_windows(values, train_end, validation_end, window_size)
test_windows = make_windows(values, validation_end, len(values), window_size)

# Ein absichtlich falsches Fenster zeigt typische Datenleckage.
leaky_input_start = train_end - window_size + 2
leaky_target_time = train_end + 2

# Kurze Ausgaben prüfen die wichtigsten Größen.
print(f"Trainingsfenster: {len(train_windows)}")
print(f"Validierungsfenster: {len(validation_windows)}")
print(f"Testfenster: {len(test_windows)}")
print(f"Falsches Ziel liegt im Validierungsbereich: {leaky_target_time}")

# Die Grafik markiert Splits und ein leckendes Fenster.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_index, values, color="black", label="Zeitreihe")
ax.axvspan(0, train_end - 1, color="tab:blue", alpha=0.12, label="Training")

ax.axvspan(train_end, validation_end - 1, color="tab:orange", alpha=0.16,
           label="Validierung")
ax.axvspan(validation_end, len(values) - 1, color="tab:green", alpha=0.12,
           label="Test")

ax.plot(range(leaky_input_start, leaky_target_time + 1),
        values[leaky_input_start:leaky_target_time + 1],
        color="red", linewidth=4, label="leckendes Fenster")

ax.set_title("Zeitlicher Split mit Beispiel für Datenleckage")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Messwert")
ax.legend(loc="upper left")
plt.show()



## **2. Kleine Sequenznetze**

### **2.1. ConvD Filter**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_02_01.jpg?v=1787665976" width="250">



>* Erkennt lokale Muster in Sequenzen
>* Teilt Parameter und bleibt CPU-effizient

>* Filterlänge bestimmt den lokalen Zeitkontext
>* Verdichtung spart Rechenzeit und stabilisiert Merkmale

>* Conv1D verarbeitet lokale Muster parallel und schnell.
>* Gut als effizienter Start, begrenzt langfristig.



In [ ]:
#@title Python-Code - ConvD Filter

# Dieses Beispiel zeigt einen kleinen Conv1D-Filter.
# Der Filter erkennt ein lokales Zeitmuster.
# Die Grafik markiert starke Filterantworten.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen eine kleine synthetische Zeitreihe.
time_steps = np.arange(40)
signal = 0.15 * np.sin(time_steps / 2.0)

# Zwei kurze Muster werden in die Sequenz eingebaut.
pattern = np.array([0.0, 0.8, 1.2, 0.8, 0.0])
signal[8:13] = signal[8:13] + pattern
signal[25:30] = signal[25:30] + pattern

# Dieser Filter sucht genau nach dem kurzen Muster.
conv_filter = pattern - np.mean(pattern)
filter_norm = np.sum(conv_filter * conv_filter)

# Die Faltung verschiebt den Filter über alle Positionen.
responses = []
for start in range(len(signal) - len(conv_filter) + 1):
    window = signal[start:start + len(conv_filter)]
    centered_window = window - np.mean(window)
    responses.append(np.sum(centered_window * conv_filter) / filter_norm)

# Wir prüfen die erwartete Länge der Filterantwort.
responses = np.array(responses)
expected_length = len(signal) - len(conv_filter) + 1

# Eine klare Prüfung hilft bei Formfehlern.
if len(responses) != expected_length:
    raise ValueError("Die Länge der Filterantwort passt nicht.")

# Die stärksten Antworten zeigen gefundene Musterpositionen.
best_positions = np.argsort(responses)[-2:][::-1]
rounded_scores = np.round(responses[best_positions], 2)

print("Conv1D-Filterlänge:", len(conv_filter))
print("Stärkste Startpositionen:", best_positions.tolist())
print("Antwortwerte dort:", rounded_scores.tolist())

# Eine Achse zeigt Signal und skalierte Filterantwort gemeinsam.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, signal, label="Zeitreihe", linewidth=2)

# Die Antwort wird zur besseren Sichtbarkeit verschoben.
response_positions = np.arange(len(responses)) + len(conv_filter) // 2
scaled_responses = responses / np.max(responses) * np.max(signal)

ax.plot(response_positions, scaled_responses, label="Filterantwort", linewidth=2)
ax.scatter(best_positions + len(conv_filter) // 2, scaled_responses[best_positions], s=80)
ax.set_title("Conv1D-Filter erkennt lokale Muster")
ax.set_xlabel("Zeitposition")

ax.set_ylabel("Signalwert und skalierte Antwort")
ax.legend()
plt.show()



### **2.2. SimpleRNN Grundlagen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_02_02.jpg?v=1787665979" width="250">



>* Verarbeitet Sequenzen mit internem Gedächtnis
>* Einfacher, CPU-freundlicher Einstieg in Zeitdaten

>* Gleiche Gewichte interpretieren jeden Zeitschritt.
>* Gut für kurze bis mittlere Muster.

>* Kleine SimpleRNNs sparen Rechenzeit
>* Zeitliche Fehler zeigen Modellgrenzen



In [ ]:
#@title Python-Code - SimpleRNN Grundlagen

# Dieses Beispiel trainiert ein kleines SimpleRNN.
# Sequenzfenster lernen den nächsten Signalwert vorherzusagen.
# Die Grafik vergleicht Vorhersage und Wahrheit.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Feste Zufallswerte machen das Ergebnis reproduzierbar.
np.random.seed(42)
tf.random.set_seed(42)

# Wir erzeugen ein kleines synthetisches Zeitsignal.
time_steps = np.arange(0, 260, dtype=np.float32)
signal = np.sin(time_steps * 0.12) + 0.25 * np.sin(time_steps * 0.03)

# Kurze Fenster bilden die Eingabe für das SimpleRNN.
window_size = 12
inputs = []
targets = []

# Jedes Fenster soll den direkt folgenden Wert vorhersagen.
for start in range(len(signal) - window_size):
    inputs.append(signal[start:start + window_size])
    targets.append(signal[start + window_size])

# TensorFlow erwartet die Form Beispiele, Zeitschritte, Merkmale.
X = np.array(inputs, dtype=np.float32).reshape(-1, window_size, 1)
y = np.array(targets, dtype=np.float32)

# Eine einfache Prüfung schützt vor falschen Fensterformen.
if X.shape[1:] != (window_size, 1):
    raise ValueError("Die Sequenzfenster haben nicht die erwartete Form.")

# Die letzten Fenster dienen als zeitlich später Testbereich.
split_index = 190
X_train = X[:split_index]
y_train = y[:split_index]

# Testdaten bleiben beim Training vollständig ungesehen.
X_test = X[split_index:]
y_test = y[split_index:]

# Das Modell bleibt klein und CPU-freundlich.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(window_size, 1)),
    tf.keras.layers.SimpleRNN(8),
    tf.keras.layers.Dense(1),
])

# Mittlerer quadratischer Fehler passt zur Zahlenvorhersage.
model.compile(optimizer="adam", loss="mse")
history = model.fit(X_train, y_train, epochs=25, batch_size=16, verbose=0)

# Wir berechnen Vorhersagen für den späteren Testabschnitt.
predictions = model.predict(X_test, verbose=0).reshape(-1)
mae = np.mean(np.abs(predictions - y_test))

# Kurze Ausgaben fassen Datenform und Fehler zusammen.
print(f"TensorFlow-Version: {tf.__version__}")
print(f"Trainingsfenster: {X_train.shape[0]}, Testfenster: {X_test.shape[0]}")
print(f"Test-MAE: {mae:.3f}")

# Eine einzelne Grafik zeigt zeitliche Fehler sichtbar.
plt.figure(figsize=(8, 4))
plt.plot(y_test, label="Wahrer nächster Wert")
plt.plot(predictions, label="SimpleRNN-Vorhersage")

# Achsenbeschriftungen machen die Sequenzinterpretation klar.
plt.title("SimpleRNN auf kurzen Sequenzfenstern")
plt.xlabel("Test-Zeitschritt")
plt.ylabel("Signalwert")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. LSTM oder GRU**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_02_03.jpg?v=1787665977" width="250">



>* LSTM und GRU speichern längere Zusammenhänge
>* Tore filtern Wichtiges und dämpfen Rauschen

>* LSTM: stärker bei komplexen langen Abhängigkeiten
>* GRU: schneller, kompakter, oft ausreichend

>* Kleine Modelle vermeiden langsames Überanpassen
>* Zeitliche Fehler zeigen Modellgrenzen



In [ ]:
#@title Python-Code - LSTM oder GRU

# Wir trainieren ein kleines GRU-Modell.
# Sequenzfenster machen Zeitreihen lernbar.
# Der Plot vergleicht Vorhersage und Wahrheit.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Feste Zufallswerte machen das Beispiel wiederholbar.
np.random.seed(42)
tf.random.set_seed(42)

# Eine synthetische Zeitreihe enthält Rhythmus und Trend.
time_steps = np.arange(260, dtype=np.float32)
season = np.sin(time_steps * 0.12)
trend = time_steps * 0.003

# Kleines Rauschen macht die Aufgabe realistischer.
rng = np.random.default_rng(42)
noise = rng.normal(0.0, 0.08, size=time_steps.shape)
series = season + trend + noise

# Die Skalierung nutzt nur den Trainingsbereich.
train_size = 200
train_mean = series[:train_size].mean()
train_std = series[:train_size].std()

# Eine kurze Prüfung verhindert ungültige Skalierung.
if train_std <= 0:
    raise ValueError("Die Trainingsstreuung muss größer als null sein.")

scaled_series = (series - train_mean) / train_std
window_size = 24

# Aus jedem Fenster entsteht ein Trainingsbeispiel.
windows = []
targets = []
for start in range(len(scaled_series) - window_size):
    end = start + window_size
    windows.append(scaled_series[start:end])
    targets.append(scaled_series[end])

# Rekurrente Schichten erwarten drei Dimensionen.
X = np.array(windows, dtype=np.float32)[..., np.newaxis]
y = np.array(targets, dtype=np.float32)

# Die zeitliche Reihenfolge bleibt beim Aufteilen erhalten.
X_train = X[: train_size - window_size]
y_train = y[: train_size - window_size]
X_test = X[train_size - window_size :]
y_test = y[train_size - window_size :]

# Eine weitere Prüfung zeigt die erwartete Fensterform.
if X_train.shape[1:] != (window_size, 1):
    raise ValueError("Die Fensterform passt nicht zum GRU-Modell.")

# Eine kleine GRU bleibt CPU-freundlich.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(window_size, 1)),
        tf.keras.layers.GRU(12),
        tf.keras.layers.Dense(1),
    ]
)

# Mean Squared Error passt zur nächsten Zahlenvorhersage.
model.compile(optimizer="adam", loss="mse")
model.fit(X_train, y_train, epochs=25, batch_size=16, verbose=0)

# Die Vorhersagen werden zurückskaliert.
pred_scaled = model.predict(X_test, verbose=0).reshape(-1)
pred = pred_scaled * train_std + train_mean
true = y_test * train_std + train_mean

# Ein naiver Vergleich nutzt den letzten Fensterwert.
naive_scaled = X_test[:, -1, 0]
naive = naive_scaled * train_std + train_mean

# Zwei Fehlerwerte zeigen den Nutzen des Sequenzmodells.
gru_mae = np.mean(np.abs(pred - true))
naive_mae = np.mean(np.abs(naive - true))

print(f"TensorFlow-Version: {tf.__version__}")
print(f"Trainingsfenster: {X_train.shape[0]}, Testfenster: {X_test.shape[0]}")
print(f"MAE naive Basislinie: {naive_mae:.3f}")
print(f"MAE kleine GRU: {gru_mae:.3f}")

# Der Plot zeigt zeitliche Fehler direkt sichtbar.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(true, label="Wahre Werte", linewidth=2)
ax.plot(pred, label="GRU-Vorhersage", linewidth=2)
ax.plot(naive, label="Naive Basislinie", linestyle="--")
ax.set_title("Kleine GRU für Ein-Schritt-Zeitreihenvorhersage")
ax.set_xlabel("Test-Zeitschritt")
ax.set_ylabel("Zeitreihenwert")
ax.legend()
plt.show()



## **3. Sequenzen fair bewerten**

### **3.1. Maskierung richtig nutzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_03_01.jpg?v=1787665981" width="250">



>* Masken ignorieren künstlich aufgefüllte Sequenzstellen
>* Nur echte Zeitpunkte fair bewerten

>* Masken verhindern verzerrte zeitliche Fehleranalysen
>* Bewerte nur echte, definierte Zielwerte

>* Masken müssen zur Fachfrage passen
>* Nur gültige Zeitpunkte fair vergleichen



In [ ]:
#@title Python-Code - Maskierung richtig nutzen

# Dieses Beispiel zeigt faire Fehlerbewertung mit Masken.
# Ungültige Auffüllstellen sollen Metriken nicht verzerren.
# Die Grafik vergleicht maskierte und unmaskierte Fehler.

import numpy as np
import matplotlib.pyplot as plt

# Drei Sequenzen werden auf gleiche Länge aufgefüllt.
true_values = np.array([
    [10.0, 12.0, 13.0, 0.0, 0.0],
    [20.0, 19.0, 18.0, 17.0, 0.0],
    [30.0, 31.0, 29.0, 28.0, 27.0],
])

# Eins bedeutet echter Zeitpunkt, null bedeutet Auffüllung.
mask = np.array([
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
], dtype=bool)

# Zwei einfache Vorhersagen simulieren verschiedene Modellfamilien.
sequence_model = np.array([
    [10.5, 11.5, 13.5, 0.0, 0.0],
    [19.0, 19.5, 18.5, 16.5, 0.0],
    [29.0, 31.5, 28.5, 28.5, 26.5],
])

signal_model = np.array([
    [9.0, 11.0, 12.0, 0.0, 0.0],
    [21.0, 20.0, 19.0, 18.0, 0.0],
    [31.0, 30.0, 30.0, 29.0, 28.0],
])

# Die Formprüfung verhindert stille Bewertungsfehler.
if true_values.shape != mask.shape:
    raise ValueError("Zielwerte und Maske brauchen dieselbe Form.")

if sequence_model.shape != true_values.shape:
    raise ValueError("Vorhersagen brauchen dieselbe Form wie Zielwerte.")

# Fehler werden einmal falsch und einmal fair berechnet.
sequence_abs_error = np.abs(sequence_model - true_values)
signal_abs_error = np.abs(signal_model - true_values)

sequence_unmasked_mae = sequence_abs_error.mean()
signal_unmasked_mae = signal_abs_error.mean()

sequence_masked_mae = sequence_abs_error[mask].mean()
signal_masked_mae = signal_abs_error[mask].mean()

# Zeitliche Fehlerkurven nutzen nur gültige Positionen.
valid_counts = mask.sum(axis=0)
sequence_time_mae = (sequence_abs_error * mask).sum(axis=0) / valid_counts
signal_time_mae = (signal_abs_error * mask).sum(axis=0) / valid_counts

print("Unmaskierter MAE Sequenznetz:", round(sequence_unmasked_mae, 3))
print("Maskierter MAE Sequenznetz:", round(sequence_masked_mae, 3))
print("Unmaskierter MAE Signalmodell:", round(signal_unmasked_mae, 3))
print("Maskierter MAE Signalmodell:", round(signal_masked_mae, 3))
print("Gültige Zeitpunkte pro Spalte:", valid_counts.tolist())

# Die Grafik zeigt faire Fehler über die Zeit.
fig, ax = plt.subplots(figsize=(7, 4))
time_steps = np.arange(1, true_values.shape[1] + 1)

ax.plot(time_steps, sequence_time_mae, marker="o", label="Sequenznetz")
ax.plot(time_steps, signal_time_mae, marker="s", label="Signalmodell")

ax.set_title("Maskierte zeitliche Fehleranalyse")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Mittlerer absoluter Fehler")

ax.set_xticks(time_steps)
ax.legend()
plt.show()



### **3.2. Zeitliche Fehleranalyse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_03_02.jpg?v=1787665983" width="250">



>* Fehlerzeitpunkte zeigen kritische Modellschwächen.
>* Zeitverlauf ermöglicht faire Modellvergleiche.

>* Fehlerverläufe zeigen verborgene Modellschwächen
>* Nach Zeit, Ereignisnähe und Fensterposition auswerten

>* Klassische Modelle stabil, Sequenznetze flexibler
>* Fehlerzeitpunkt entscheidet über praktischen Nutzen



In [ ]:
#@title Python-Code - Zeitliche Fehleranalyse

# Wir untersuchen Fehler entlang einer Zeitreihe.
# Zwei einfache Modelle reagieren unterschiedlich schnell.
# Die Grafik zeigt kritische Fehlerphasen.

import numpy as np
import matplotlib.pyplot as plt

# Eine kleine synthetische Zeitreihe bleibt vollständig reproduzierbar.
rng = np.random.default_rng(42)
time_steps = np.arange(120)

# Das Signal enthält Rhythmus, Trend und einen plötzlichen Sprung.
seasonal = 2.0 * np.sin(time_steps / 6.0)
trend = 0.03 * time_steps

jump = np.where(time_steps >= 70, 4.0, 0.0)
noise = rng.normal(0.0, 0.25, size=time_steps.size)
actual = 10.0 + seasonal + trend + jump + noise

# Modell A ist ein träger gleitender Mittelwert.
window = 8
moving_average = np.empty_like(actual)

for index in range(time_steps.size):
    start = max(0, index - window)
    moving_average[index] = np.mean(actual[start:index + 1])

# Modell B reagiert schneller auf den letzten Messwert.
fast_filter = np.empty_like(actual)
fast_filter[0] = actual[0]

for index in range(1, time_steps.size):
    fast_filter[index] = 0.75 * actual[index - 1] + 0.25 * fast_filter[index - 1]

# Wir vergleichen absolute Fehler global und im Sprungbereich.
ma_error = np.abs(actual - moving_average)
fast_error = np.abs(actual - fast_filter)

critical_mask = (time_steps >= 70) & (time_steps <= 82)
ma_mean = np.mean(ma_error)
fast_mean = np.mean(fast_error)

ma_critical = np.mean(ma_error[critical_mask])
fast_critical = np.mean(fast_error[critical_mask])

print(f"Mittlerer Fehler gleitender Mittelwert: {ma_mean:.2f}")
print(f"Mittlerer Fehler schneller Filter: {fast_mean:.2f}")
print(f"Fehler im Sprungbereich gleitender Mittelwert: {ma_critical:.2f}")
print(f"Fehler im Sprungbereich schneller Filter: {fast_critical:.2f}")
print("Die zeitliche Sicht zeigt, wann ein Modell schwächelt.")

# Eine einzelne Achse zeigt Signal und Fehlerphase gemeinsam.
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(time_steps, actual, label="Tatsächlicher Wert", color="black")
ax.plot(time_steps, moving_average, label="Gleitender Mittelwert", color="tab:blue")
ax.plot(time_steps, fast_filter, label="Schneller Filter", color="tab:orange")
ax.axvspan(70, 82, color="red", alpha=0.15, label="kritischer Übergang")

ax.set_title("Zeitliche Fehleranalyse bei einem plötzlichen Übergang")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Signalwert")
ax.legend()
plt.show()



### **3.3. Sequenzen vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_03_03.jpg?v=1787665985" width="250">



>* Modelle über den Zeitverlauf vergleichen
>* Fehlerprofile bei Mustern und Änderungen beachten

>* Gleiche Fenster, Ziele und Masken nutzen
>* Fehler zeitlich und nach Signalphasen prüfen

>* Klassische Modelle: transparent, schnell und robust
>* Sequenznetze: komplexer, datenhungriger, fallabhängig besser



In [ ]:
#@title Python-Code - Sequenzen vergleichen

# Wir vergleichen Sequenzfehler über die Zeit.
# Zwei einfache Modelle reagieren unterschiedlich schnell.
# Die Grafik zeigt Fehler und Verzögerung.

import numpy as np
import matplotlib.pyplot as plt

# Ein deterministisches Signal simuliert Temperaturwerte in Grad Celsius.
time_steps = np.arange(80)
trend = 0.03 * time_steps
season = 1.2 * np.sin(time_steps / 5)

# Ein plötzlicher Sprung prüft die zeitliche Reaktionsfähigkeit.
step_change = np.where(time_steps >= 45, 3.0, 0.0)
true_signal = 20 + trend + season + step_change

# Das gleitende Mittel ist stabil, aber oft verzögert.
window_size = 5
moving_average = np.convolve(true_signal, np.ones(window_size) / window_size, mode="same")

# Das reaktive Modell folgt Änderungen schneller, aber glättet weniger.
reactive_model = np.empty_like(true_signal)
reactive_model[0] = true_signal[0]
for index in range(1, len(true_signal)):
    reactive_model[index] = 0.35 * true_signal[index] + 0.65 * reactive_model[index - 1]

# Randwerte werden fair ausgeschlossen, weil das Fenster dort unvollständig ist.
valid_mask = np.ones_like(true_signal, dtype=bool)
valid_mask[:2] = False
valid_mask[-2:] = False

# Fehler werden auf denselben Zeitpunkten verglichen.
mae_average = np.mean(np.abs(moving_average[valid_mask] - true_signal[valid_mask]))
mae_reactive = np.mean(np.abs(reactive_model[valid_mask] - true_signal[valid_mask]))

# Der Übergangsbereich zeigt, welches Modell schneller reagiert.
transition_mask = (time_steps >= 45) & (time_steps <= 55)
transition_error_average = np.mean(
    np.abs(moving_average[transition_mask] - true_signal[transition_mask])
)
transition_error_reactive = np.mean(
    np.abs(reactive_model[transition_mask] - true_signal[transition_mask])
)

print(f"MAE gleitendes Mittel: {mae_average:.2f} °C")
print(f"MAE reaktives Modell: {mae_reactive:.2f} °C")
print(f"Übergangsfehler Mittel: {transition_error_average:.2f} °C")
print(f"Übergangsfehler reaktiv: {transition_error_reactive:.2f} °C")

# Eine einzelne Achse zeigt Signal und Modellverhalten gemeinsam.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, true_signal, label="Wahrer Verlauf", linewidth=2)
ax.plot(time_steps, moving_average, label="Gleitendes Mittel")

ax.plot(time_steps, reactive_model, label="Reaktives Modell")
ax.axvspan(45, 55, color="orange", alpha=0.15, label="Übergang")
ax.set_title("Sequenzen fair vergleichen: Gesamtfehler und Übergang")

ax.set_xlabel("Zeitschritt")
ax.set_ylabel("Temperatur in °C")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Sequenzmodelle**</font>


In this lecture, you learned to:
- Bereiten Sequenzfenster und Baselines für zeitliche Vorhersage- oder Klassifikationsaufgaben vor. 
- Trainieren kleine Conv1D-, SimpleRNN- und LSTM- oder GRU-Modelle CPU-freundlich. 
- Vergleichen Sequenznetze mit klassischen Signalmodellen anhand zeitlicher Fehleranalysen. 

In the next Lecture (Lecture B), we will go over 'Autoencoder und Generierung'